# Suricata IDS/IPS Alert Datamap — reproducible build

Builds an interactive datamap of Suricata `eve.json` alert logs from four captures
(WRCCDC 2017, WRCCDC 2018, FIRST.org conference 2015, an internet honeypot 2018),
plus a static preview PNG.

**Pipeline:** parse `eve.json` → composite text + dedup → `build` (embed → layout →
hierarchical clusters) → **hand-written cluster labels** → `render` → interactive HTML.

Two things in this corpus will silently ruin the map if you change them. Both are
covered in the *Limitations and traps* section at the bottom — read it before editing
anything above.

Source data: <https://github.com/FrankHassanabad/suricata-sample-data/releases/tag/v4.0.0>

In [1]:
# ---- configuration ------------------------------------------------------
import os

# Folder holding the four extracted capture directories and the deliverables.
BASE = os.environ.get('SURICATA_DATAMAP_DIR', '/sessions/happy-clever-brown/mnt/Suricata - Datamap')

SKILL_SRC = os.environ.get('DATAMAP_SKILL_DIR', '/sessions/happy-clever-brown/mnt/.claude/skills/datamap-builder')
WORK_DMB  = '/tmp/work/dmb'                 # writable copy of the skill (patched below)
PIPELINE  = f'{WORK_DMB}/scripts/datamap_pipeline.py'

CORPUS    = '/tmp/suricata_alerts_corpus.csv'
PARTS_DIR = '/tmp/corpus_parts'
WORKDIR   = '/tmp/dm_suricata'              # build caches; one per dataset
TEXT_COL  = 'text'

LAYERS      = 3
SEED        = 42
CLUSTER_KS  = '12,48,144'                   # coarse -> fine; see Limitations
DATE        = '2026-08-12'

OUT_HTML  = f'/tmp/suricata_alert_datamap_{DATE}.html'
TITLE     = 'Suricata IDS/IPS Alerts'
DARKMODE  = True
COLOR_COL = 'source'
TEMPLATE  = f'{BASE}/suricata_datamap_template.html'   # patched click-to-open-card viewer

# Tooltip fields (the stock hover string) and the structured fields the info card reads.
HOVER_COLS  = 'signature,category,severity,src_ip,dest_ip,dest_port,app_proto,count,timestamp'
RECORD_COLS = ('signature,signature_id,category,severity,src_ip,src_port,dest_ip,dest_port,'
               'app_proto,hostname,uri,count,timestamp,source')

# Click-through: the info card turns each point's Suricata rule id into a link to the
# public Emerging Threats rule page. {sid} is substituted per point.
SID_LOOKUP_URL = 'https://threatintel.proofpoint.com/sid/{sid}'

# Re-parsing the two 600-800 MB eve.json files takes ~80 s. The extraction is fully
# deterministic, so by default we reuse an existing corpus. Set True to force a reparse.
FORCE_EXTRACT = bool(int(os.environ.get('FORCE_EXTRACT', '0')))

# Sources: (directory, canonical name, sample target after dedup; None = keep all)
SOURCES = [
    ('honeypot-2018',      'honeypot-2018',  None),
    ('first-org-conf-2015','first-org-2015', None),
    ('wrcddc-2017',        'wrccdc-2017',    27500),
    ('wrcddc-2018',        'wrccdc-2018',    27500),
]
print('BASE       :', BASE)
print('exists     :', os.path.isdir(BASE))
for d, name, _ in SOURCES:
    p = os.path.join(BASE, d, 'eve.json')
    alt = os.path.join(BASE, 'release', 'release', d, 'eve.json')
    hit = p if os.path.exists(p) else (alt if os.path.exists(alt) else None)
    print(f'  {name:16s}', f'{os.path.getsize(hit)/1e6:8.1f} MB' if hit else 'MISSING')

BASE       : /sessions/happy-clever-brown/mnt/Suricata - Datamap
exists     : True
  honeypot-2018         0.1 MB
  first-org-2015        6.6 MB
  wrccdc-2017         663.8 MB
  wrccdc-2018         838.1 MB


In [2]:
# ---- environment --------------------------------------------------------
# The full engine (UMAP + HDBSCAN + datamapplot) is not installable in this sandbox:
# there is no outbound network, so pip cannot reach PyPI. The pipeline detects this and
# falls back to its numpy-only LITE engine (PCA + neighbour refinement layout, k-means
# hierarchy) and the zero-dependency custom HTML renderer. That fallback is expected
# here, not an error -- but it does mean the layout is less organic than a real UMAP,
# and every cluster id below is tied to the lite engine.
import importlib, sys
for m in ['numpy', 'pandas', 'matplotlib', 'sklearn', 'umap', 'hdbscan', 'datamapplot']:
    try:
        importlib.import_module(m)
        print(f'  {m:14s} available')
    except Exception:
        print(f'  {m:14s} NOT available')
print('\npython', sys.version.split()[0])

  numpy          available
  pandas         available
  matplotlib     available
  sklearn        NOT available
  umap           NOT available
  hdbscan        NOT available
  datamapplot    NOT available

python 3.10.12


## 1. Build the corpus

Each `eve.json` is newline-delimited JSON, one event per line, streamed with the stdlib
`json` module — never loaded whole (the two WRCCDC files are 664 MB and 838 MB).

**The text we embed is a composite, not the signature.** Measured on the full files,
this corpus has only 43 / 135 / 161 / 206 distinct signatures per source. Embedding
`signature` alone collapses the map to a few hundred stacked points. Folding in
`category`, `app_proto`, `hostname`, the URI path and `dest_port` is what produces tens
of thousands of distinct positions.

**But raw URIs manufacture fake variety.** See `normalize_uri` below and the Limitations
section — this is the single most important line in the notebook.

In [3]:
# ---- extraction ---------------------------------------------------------
import json, os, re, csv, random
from collections import Counter

_DIGIT_RUN = re.compile(r'\d{4,}')
_HEX_BLOB  = re.compile(r'(?i)\b[0-9a-f]{12,}\b')

def normalize_uri(uri):
    '''Collapse nonce-like tokens so they don't manufacture fake variety.

    The NETGEAR WNR2000v5 exploit URI is always
        /apply_noauth.cgi?/lang_check.html%20timestamp=<random 8 digits>
    Left raw, 22,173 alerts look like 22,173 distinct composites -- 37% of the corpus --
    when they are really 21 distinct alerts (same signature, same path, ~10 target
    hosts). Dedup cannot see through the nonce, so a third of the map ends up devoted
    to one attack shattered into meaningless blobs.

    Applied to the composite text / dedup key ONLY; the raw `uri` column is preserved
    for display in the hover tooltip and the info card.
    '''
    return _HEX_BLOB.sub('H', _DIGIT_RUN.sub('N', str(uri)))

def composite(sig, cat, app_proto, hostname, uri, dest_port):
    parts = [str(v) for v in (sig, cat, app_proto, hostname) if v]
    if uri:
        parts.append(normalize_uri(uri)[:120])
    if dest_port not in (None, '', 0):
        parts.append(str(dest_port))
    return ' | '.join(parts)

FIELDS = ['text','signature','signature_id','category','severity','proto','app_proto',
          'src_ip','src_port','dest_ip','dest_port','hostname','uri','timestamp',
          'source','count']

def resolve(dirname):
    for p in (os.path.join(BASE, dirname, 'eve.json'),
              os.path.join(BASE, 'release', 'release', dirname, 'eve.json')):
        if os.path.exists(p):
            return p
    return None

def extract_source(dirname, source_name, target):
    '''Stream one eve.json -> deduplicated per-source CSV. Returns stats.'''
    path = resolve(dirname)
    if path is None:
        raise FileNotFoundError(f'no eve.json for {source_name}')
    dedup, counts, signatures = {}, Counter(), set()
    n_lines = n_alerts = 0
    with open(path, 'r', errors='replace') as fh:
        for line in fh:
            n_lines += 1
            if '"alert"' not in line:          # cheap prefilter before json.loads
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            if rec.get('event_type') != 'alert':
                continue
            n_alerts += 1
            alert = rec.get('alert') or {}
            sig   = alert.get('signature') or ''
            signatures.add(sig)
            http  = rec.get('http') or {}
            tls   = rec.get('tls') or {}
            dns   = rec.get('dns') or {}
            hostname  = http.get('hostname') or tls.get('sni') or dns.get('rrname') or ''
            uri       = http.get('url') or ''
            app_proto = rec.get('app_proto') or ''
            dest_port = rec.get('dest_port')
            text = composite(sig, alert.get('category') or '', app_proto,
                             hostname, uri, dest_port)
            key = (text, dest_port)
            counts[key] += 1
            if key not in dedup:               # keep first occurrence
                dedup[key] = {
                    'text': text, 'signature': sig,
                    # Suricata rule id. Not part of the composite text (it is a
                    # restatement of the signature name, so it would add no spread),
                    # but it is the key that resolves to a public rule-intel page.
                    'signature_id': alert.get('signature_id') if alert.get('signature_id') is not None else '',
                    'category': alert.get('category') or '',
                    'severity': alert.get('severity') if alert.get('severity') is not None else '',
                    'proto': rec.get('proto') or '', 'app_proto': app_proto,
                    'src_ip': rec.get('src_ip') or '',
                    'src_port': rec.get('src_port') if rec.get('src_port') is not None else '',
                    'dest_ip': rec.get('dest_ip') or '',
                    'dest_port': dest_port if dest_port is not None else '',
                    'hostname': hostname, 'uri': uri,
                    'timestamp': rec.get('timestamp') or '', 'source': source_name,
                }
    keys = sorted(dedup, key=lambda k: (str(k[0]), str(k[1])))   # deterministic
    if target is not None and len(keys) > target:
        keys = sorted(random.Random(SEED).sample(keys, target),
                      key=lambda k: (str(k[0]), str(k[1])))
    os.makedirs(PARTS_DIR, exist_ok=True)
    out_path = os.path.join(PARTS_DIR, f'{source_name}.csv')
    with open(out_path, 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS); w.writeheader()
        for k in keys:
            row = dict(dedup[k]); row['count'] = counts[k]; w.writerow(row)
    return {'source': source_name, 'lines': n_lines, 'alerts': n_alerts,
            'unique_composites': len(dedup), 'unique_signatures': len(signatures),
            'rows_kept': len(keys)}
print('extraction helpers defined')

extraction helpers defined


In [4]:
# ---- run extraction -----------------------------------------------------
import time
need = FORCE_EXTRACT or not os.path.exists(CORPUS) or not all(
    os.path.exists(os.path.join(PARTS_DIR, f'{n}.csv')) for _, n, _ in SOURCES)

if need:
    stats = []
    for dirname, name, target in SOURCES:
        t0 = time.time()
        s = extract_source(dirname, name, target)
        s['seconds'] = round(time.time() - t0, 1)
        stats.append(s)
        print(f"{s['source']:16s} lines={s['lines']:>9,} alerts={s['alerts']:>9,} "
              f"uniq_composites={s['unique_composites']:>7,} "
              f"uniq_signatures={s['unique_signatures']:>4,} kept={s['rows_kept']:>6,} "
              f"({s['seconds']}s)")
    json.dump(stats, open('/tmp/extract_stats.json', 'w'), indent=2)
else:
    stats = json.load(open('/tmp/extract_stats.json'))
    print('reusing existing per-source CSVs (set FORCE_EXTRACT=1 to reparse)\n')
    for s in stats:
        print(f"{s['source']:16s} alerts={s['alerts']:>9,} "
              f"uniq_composites={s['unique_composites']:>7,} "
              f"uniq_signatures={s['unique_signatures']:>4,} kept={s['rows_kept']:>6,}")

reusing existing per-source CSVs (set FORCE_EXTRACT=1 to reparse)

honeypot-2018    alerts=      254 uniq_composites=    150 uniq_signatures=  43 kept=   150
first-org-2015   alerts=    7,750 uniq_composites=  4,429 uniq_signatures= 135 kept= 4,429
wrccdc-2017      alerts=  755,711 uniq_composites=120,749 uniq_signatures= 161 kept=27,500
wrccdc-2018      alerts=1,052,233 uniq_composites= 89,068 uniq_signatures= 206 kept=27,500


In [5]:
# ---- combine into the corpus CSV ----------------------------------------
import pandas as pd

frames = []
for _, name, _ in [('','wrccdc-2017',0), ('','wrccdc-2018',0),
                   ('','first-org-2015',0), ('','honeypot-2018',0)]:
    frames.append(pd.read_csv(os.path.join(PARTS_DIR, f'{name}.csv'), keep_default_na=False,
                              dtype={'severity':'object','src_port':'object','dest_port':'object'}))
df = pd.concat(frames, ignore_index=True)

ts = pd.to_datetime(df['timestamp'], format='mixed', utc=True, errors='coerce')
df['date'] = ts.dt.date.astype(str)
df.loc[ts.isna(), 'date'] = ''
df = df.sort_values(['source', 'text'], kind='mergesort').reset_index(drop=True)
df.to_csv(CORPUS, index=False)

print(f'{CORPUS}: {len(df):,} rows, {os.path.getsize(CORPUS)/1e6:.1f} MB')
print(df['source'].value_counts().to_string())
print(f"\nunique composite texts : {df['text'].nunique():,}")
print(f"unique signatures      : {df['signature'].nunique():,}")
print(f"raw alerts represented : {df['count'].astype(int).sum():,}")
print(f"rows carrying host/URI : {((df['hostname']!='')|(df['uri']!='')).mean():.1%}")
df.head(3)[['signature','hostname','dest_port','source','count']]

/tmp/suricata_alerts_corpus.csv: 59,579 rows, 20.0 MB
source
wrccdc-2018       27500
wrccdc-2017       27500
first-org-2015     4429
honeypot-2018       150

unique composite texts : 58,787
unique signatures      : 293
raw alerts represented : 496,578
rows carrying host/URI : 47.2%
                                           signature  ... count
0  ET ATTACK_RESPONSE Output of id command from H...  ...     5
1                   ET CHAT Facebook Chat using XMPP  ...    33
2                  ET CHAT Skype User-Agent detected  ...     1

[3 rows x 5 columns]


## 2. Patch the pipeline and install the viewer template

Four deviations from the stock skill, all required for this dataset:

1. **Columnar, dictionary-encoded payload (`cols` / `fine` / `colors`) + `legend`** — the
   info card reads structured per-point fields instead of re-parsing a hover string, and
   the legend makes the colour facet readable. Each column ships once as
   `{d: distinct values, i: index per point}` where that is smaller. The stock row-of-dicts
   `records` block plus 59,579 pre-joined `hover` strings were 29 MB of a 34 MB file, nearly
   all repetition (`category` has 17 distinct values, `source` 4, `signature` 293).
   Tooltips are rebuilt in the browser from `hover_cols`, so the joined strings are gone.
2. **`--palette`** — the default `tab20` assignment gives honeypot-2018 (150 points,
   0.25% of the map) a pale blue that disappears. It gets hot pink instead.
3. **Lite-engine performance + `--cluster-ks`** — the lite engine full-`argsort`s an
   n-wide similarity row per point and materialises an `(n, k, 2)` tensor per k-means
   iteration. At n≈60k that does not finish. Replaced with `argpartition` and the
   `|x|²-2x·C+|C|²` expansion (same results, ~50x faster). `--cluster-ks` overrides the
   n-derived heuristic, which would otherwise ask for 595 / 1,489 / 3,971 clusters —
   unlabelable by hand and illegible on screen.
4. **Preview framing** — clamp the PNG axes to the 0.5–99.5 percentile range so a few
   layout outliers don't squash 99% of the points into a corner.

**The template copy is not optional.** `render_custom()` hardcodes
`<skill>/assets/custom_template.html`, so the patched viewer must be copied into place
*before* rendering or you silently get the stock viewer with no info cards.

In [6]:
# ---- patch a writable copy of the skill ---------------------------------
import json, os, shutil, subprocess, sys

def _make_writable(path):
    # copytree preserves the skills mount's read-only bits, which would make a later
    # rmtree fail; chmod the whole tree before touching it.
    for root, dirs, files in os.walk(path):
        for n in dirs + files:
            try: os.chmod(os.path.join(root, n), 0o755)
            except OSError: pass
    try: os.chmod(path, 0o755)
    except OSError: pass

if os.path.isdir(WORK_DMB):
    _make_writable(WORK_DMB)
    shutil.rmtree(WORK_DMB)
os.makedirs(os.path.dirname(WORK_DMB), exist_ok=True)
shutil.copytree(SKILL_SRC, WORK_DMB)
_make_writable(WORK_DMB)

src = open(PIPELINE, encoding='utf-8').read()

def sub(old, new, what):
    global src
    assert old in src, f'anchor not found: {what}'
    src = src.replace(old, new, 1)

# (1) palette override -----------------------------------------------------
sub('''        palette = _categorical_palette(sorted(set(cats)))
''',
'''        palette = _categorical_palette(sorted(set(cats)))
        override_path = getattr(args, "palette", None)
        if override_path:
            with open(override_path, encoding="utf-8") as _pf:
                palette.update(json.load(_pf))
            log(f"palette overridden from {override_path}")
''', 'palette')

# (2) columnar, dictionary-encoded payload ---------------------------------
# A row-of-dicts `records` block plus 59,579 pre-joined `hover` strings came to
# 29 MB of a 34 MB file, most of it repetition: `category` has 17 distinct values,
# `source` has 4, `signature` has 293. Each column is emitted once as
# {"d": distinct values, "i": index per point} when that is smaller, else {"v": raw}.
# Tooltips are rebuilt in the browser from `hover_cols`, so the joined strings go away.
sub('''    payload = {
        "title": args.title or "Datamap",''',
'''    def _encode_column(values):
        # Dictionary-encode when it pays; otherwise store raw.
        distinct = list(dict.fromkeys(values))
        if len(distinct) <= max(2, len(values) // 2):
            key = {v: k for k, v in enumerate(distinct)}
            enc = {"d": distinct, "i": [key[v] for v in values]}
            if len(json.dumps(enc)) < len(json.dumps({"v": values})):
                return enc, True
        return {"v": values}, False

    def _clean(v):
        if v is None or (isinstance(v, float) and v != v):
            return ""
        return v if isinstance(v, (int, float, str)) else str(v)

    rec_spec = getattr(args, "record_cols", None) or args.hover_cols or ""
    rec_cols = [c.strip() for c in rec_spec.split(",") if c.strip() and c.strip() in df.columns]
    cols_payload, n_dict = {}, 0
    for c in rec_cols:
        enc, was_dict = _encode_column([_clean(v) for v in df[c].tolist()])
        cols_payload[c] = enc
        n_dict += int(was_dict)
    if rec_cols:
        log(f"columnar payload: {len(df):,} points x {len(rec_cols)} fields "
            f"({n_dict} dictionary-encoded)")

    hover_only = [c.strip() for c in (args.hover_cols or "").split(",")
                  if c.strip() and c.strip() in df.columns]
    fine_enc, _ = _encode_column([str(x) for x in vectors[-1]])
    colors_enc = None
    if cats:
        colors_enc, _ = _encode_column([palette.get(c, "#8888aa") for c in cats])

    legend = None
    if cats:
        seen = {}
        for c in cats:
            seen[c] = seen.get(c, 0) + 1
        legend = [{"label": c, "color": palette.get(c, "#8888aa"), "n": seen[c]} for c in sorted(seen)]

    # Click-through: a per-point external lookup rendered as a link inside the info
    # card. {v} is replaced with that point's value from the named column. Nothing
    # navigates on a plain point click -- the user has to click the link itself.
    card_link = None
    _lcol = getattr(args, "card_link_col", None)
    if _lcol and _lcol in df.columns and getattr(args, "card_link_url", None):
        card_link = {"col": _lcol, "url": args.card_link_url,
                     "label": getattr(args, "card_link_label", None) or _lcol}
        _cov = float((df[_lcol].astype(str).str.strip() != "").mean())
        log(f"card link: {_lcol} -> {args.card_link_url} ({_cov*100:.1f}% of points)")

    payload = {
        "cols": cols_payload,
        "card_link": card_link,
        "hover_cols": hover_only,
        "hover_max_chars": args.hover_max_chars,
        "fine": fine_enc,
        "legend": legend,
        "legend_title": args.color_col,
        "title": args.title or "Datamap",''', 'payload')

# (2b) drop the three now-redundant bulk keys -------------------------------
sub('''        "hover": hover,
        "fine_labels": list(vectors[-1]),
        "label_layers": label_layers,
        "colors": [palette.get(c, "#8888aa") for c in cats] if cats else None,
        "urls": urls,
    }''',
'''        "label_layers": label_layers,
        "colors": colors_enc,
        "urls": urls,
    }''', 'drop redundant keys')

# (3) new render CLI args --------------------------------------------------
sub('''    r.add_argument("--hover-max-chars", type=int, default=300)
''',
'''    r.add_argument("--hover-max-chars", type=int, default=300)
    r.add_argument("--record-cols", default=None)
    r.add_argument("--palette", default=None)
    r.add_argument("--card-link-col", default=None,
                   help="column whose value builds a per-point lookup link in the info card")
    r.add_argument("--card-link-url", default=None,
                   help="URL template for --card-link-col; {v} is the column value")
    r.add_argument("--card-link-label", default=None)
''', 'render args')

# (4) lite kNN: argpartition instead of a full argsort ---------------------
sub('''        nbrs[s:s + chunk] = np.argsort(-sims, axis=1)[:, :k]
''',
'''        part = np.argpartition(-sims, k, axis=1)[:, :k]
        rows_ = np.arange(part.shape[0])[:, None]
        nbrs[s:s + chunk] = part[rows_, np.argsort(-sims[rows_, part], axis=1)]
''', 'knn')

# (5) fast k-means ---------------------------------------------------------
sub('''    centers = [coords[rng.integers(len(coords))]]
    for _ in range(k - 1):
        d2 = np.min([( (coords - c) ** 2).sum(axis=1) for c in centers], axis=0)
        probs = d2 / max(d2.sum(), 1e-12)
        centers.append(coords[rng.choice(len(coords), p=probs)])
    C = np.array(centers)
    for _ in range(iters):
        d = ((coords[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
        assign = d.argmin(axis=1)
        newC = np.array([coords[assign == j].mean(axis=0) if np.any(assign == j) else C[j]
                         for j in range(k)])''',
'''    X = np.ascontiguousarray(coords, dtype=np.float64)
    n_ = len(X)
    C = np.empty((k, X.shape[1]), dtype=np.float64)
    C[0] = X[rng.integers(n_)]
    d2 = ((X - C[0]) ** 2).sum(axis=1)
    for j in range(1, k):                      # k-means++ with a running minimum
        tot = d2.sum()
        C[j] = X[rng.integers(n_)] if (not np.isfinite(tot) or tot <= 1e-12) \\
               else X[rng.choice(n_, p=d2 / tot)]
        np.minimum(d2, ((X - C[j]) ** 2).sum(axis=1), out=d2)
    assign = np.zeros(n_, dtype=np.int64)
    for _ in range(iters):
        d = (-2.0) * (X @ C.T) + (C ** 2).sum(axis=1)[None, :]
        assign = d.argmin(axis=1)
        cnt = np.bincount(assign, minlength=k).astype(np.float64)
        sums = np.zeros_like(C)
        for dim in range(X.shape[1]):
            sums[:, dim] = np.bincount(assign, weights=X[:, dim], minlength=k)
        ne = cnt > 0
        newC = C.copy()
        newC[ne] = sums[ne] / cnt[ne, None]''', 'kmeans')

# (6) --cluster-ks ---------------------------------------------------------
sub('''def lite_hierarchical_clusters(coords, n_layers=3, seed=42):
    n = len(coords)
    ks = {1:''',
'''def lite_hierarchical_clusters(coords, n_layers=3, seed=42, cluster_ks=None):
    n = len(coords)
    if cluster_ks:
        ks = list(cluster_ks)[:max(1, min(3, n_layers))]
        return _lite_km_layers(coords, ks, seed, n)
    ks = {1:''', 'hierarchy')
sub('''def lite_ctfidf_keywords(''',
'''def _lite_km_layers(coords, ks, seed, n):
    ks = [min(k, n // 5) if n >= 25 else 3 for k in ks]
    layers = []
    for k in ks:
        layers.append(_kmeans(coords, int(max(2, k)), seed))
        log(f"  lite layer: k-means k={int(max(2, k))}")
    return layers, ks


def lite_ctfidf_keywords(''', 'km layers helper')
sub('''    cl_key = param_hash(lay_key, args.layers, engine, "cluster-v1")''',
'''    cluster_ks = getattr(args, "cluster_ks", None)
    if isinstance(cluster_ks, str):
        cluster_ks = [int(x) for x in cluster_ks.split(",") if x.strip()]
    cl_key = param_hash(lay_key, args.layers, engine, "cluster-v1",
                        ",".join(map(str, cluster_ks)) if cluster_ks else "auto")''', 'cluster key')
sub('''        layer_labels, sizes = lite_hierarchical_clusters(coords, args.layers, args.seed)''',
'''        layer_labels, sizes = lite_hierarchical_clusters(coords, args.layers, args.seed,
                                                         cluster_ks=cluster_ks)''', 'cluster call')
sub('''    b.add_argument("--umap-neighbors", type=int, default=15)''',
'''    b.add_argument("--cluster-ks", default=None)
    b.add_argument("--umap-neighbors", type=int, default=15)''', 'build arg')

# (7) record the extra knobs + the viewer template in the manifest ---------
sub('''            "sidecar": args.sidecar, "labels_file": args.labels,
''',
'''            "sidecar": args.sidecar, "labels_file": args.labels,
            "record_cols": getattr(args, "record_cols", None),
            "palette": getattr(args, "palette", None),
            "template": (os.path.join(os.path.dirname(os.path.dirname(
                os.path.abspath(__file__))), "assets", "custom_template.html")
                if args.mode == "custom" else None),
''', 'manifest render_params')
sub('''            "umap_neighbors": args.umap_neighbors, "umap_min_dist": args.umap_min_dist,
''',
'''            "umap_neighbors": args.umap_neighbors, "umap_min_dist": args.umap_min_dist,
            "cluster_ks": getattr(args, "cluster_ks", None),
''', 'manifest build_params')

# (8) HTML-safe payload injection -- THE bug that made the map unopenable ---
# render_custom() injects json.dumps(payload) raw into <script>...</script>.
# json.dumps does not escape '<', and this corpus is full of XSS-probe URIs that
# contain a literal '</script>'. The HTML parser ends the script element at the
# FIRST one (~1% into the file), so the viewer never initialises and the browser
# renders the remaining ~33 MB of JSON as text. Escaping < > & as \uXXXX keeps the
# JSON semantically identical while making it inert to the HTML tokenizer.
sub('''    html = template.replace("/*__DATAMAP_DATA__*/null", json.dumps(payload))''',
'''    _json = (json.dumps(payload)
             .replace("<", "\\\\u003c")
             .replace(">", "\\\\u003e")
             .replace("&", "\\\\u0026")
             .replace("\\u2028", "\\\\u2028")
             .replace("\\u2029", "\\\\u2029"))
    html = template.replace("/*__DATAMAP_DATA__*/null", _json)''', 'html-safe payload')

# (9) preview framing ------------------------------------------------------
sub('''        ax.set_title(args.title or "Datamap", color=fg, fontsize=16, fontweight="bold")''',
'''        if len(coords) > 200:
            _x0, _x1 = np.percentile(coords[:, 0], [0.5, 99.5])
            _y0, _y1 = np.percentile(coords[:, 1], [0.5, 99.5])
            if _x1 > _x0 and _y1 > _y0:
                _mx, _my = (_x1 - _x0) * 0.05, (_y1 - _y0) * 0.05
                ax.set_xlim(_x0 - _mx, _x1 + _mx); ax.set_ylim(_y0 - _my, _y1 + _my)
        ax.set_title(args.title or "Datamap", color=fg, fontsize=16, fontweight="bold")''', 'preview frame')

open(PIPELINE, 'w', encoding='utf-8').write(src)
compile(src, PIPELINE, 'exec')
print('pipeline patched and compiles')

# the patched viewer must land in assets/ BEFORE render
shutil.copy(TEMPLATE, f'{WORK_DMB}/assets/custom_template.html')
assert '__DATAMAP_DATA__' in open(f'{WORK_DMB}/assets/custom_template.html', encoding='utf-8').read()
print('viewer template installed:', TEMPLATE)

PALETTE = '/tmp/palette.json'
json.dump({'wrccdc-2017': '#4cc9f0', 'wrccdc-2018': '#b388ff',
           'first-org-2015': '#ffd166', 'honeypot-2018': '#ff2e63'},
          open(PALETTE, 'w'), indent=2)
print('palette written:', PALETTE)

pipeline patched and compiles
viewer template installed: /sessions/happy-clever-brown/mnt/Suricata - Datamap/suricata_datamap_template.html
palette written: /tmp/palette.json


### Pin the layout and clusters — do not skip this

The lite engine is **not reproducible across processes.** `lite_embed` is deterministic
*within* a run, but its randomized SVD goes through multithreaded BLAS, and the reduction
order changes with thread scheduling. A rerun on the same bytes measured a max embedding
delta of 1.52, which rotates the layout and reshuffles every k-means cluster id. Verified
the hard way: a rebuild produced only **5% agreement** with the previous per-point labels.

So the labels above are not pinned to the *parameters*, they are pinned to the *artifacts*.
`layout_coords.npy` and `cluster_assignments.npz` ship alongside this notebook and are
planted into the build cache below, keyed to the current corpus hash. That makes the map
reproducible and, just as importantly, keeps the layout stable when the corpus gains a
column — which is exactly what happened when `signature_id` was added for click-through.

Delete those two files and the next build silently relabels the whole map.

In [7]:
# ---- pin layout + clusters into the build cache -------------------------
import importlib.util, hashlib
spec = importlib.util.spec_from_file_location('dmp', PIPELINE)
dmp = importlib.util.module_from_spec(spec); spec.loader.exec_module(dmp)

PIN_LAYOUT   = f'{BASE}/layout_coords.npy'
PIN_CLUSTERS = f'{BASE}/cluster_assignments.npz'
cache_dir = f'{WORKDIR}/cache'
os.makedirs(cache_dir, exist_ok=True)

if os.path.exists(PIN_LAYOUT) and os.path.exists(PIN_CLUSTERS):
    import numpy as np
    # recompute the same cache keys cmd_build() will look for, for THIS corpus
    data_hash = dmp.sha256_file(CORPUS, extra=TEXT_COL.encode())
    emb_key = dmp.param_hash(data_hash, None, SEED, 'auto', 'lite', 'emb-v1')
    lay_key = dmp.param_hash(emb_key, 15, 0.1, SEED, 'lite', 'layout-v1')
    cl_key  = dmp.param_hash(lay_key, LAYERS, 'lite', 'cluster-v1', CLUSTER_KS)

    coords = np.load(PIN_LAYOUT).astype(np.float32)
    np.savez_compressed(f'{cache_dir}/layout_{lay_key}.npz', coords=coords)
    z = np.load(PIN_CLUSTERS)
    np.savez_compressed(f'{cache_dir}/clusters_{cl_key}.npz',
                        **{k: z[k] for k in z.files})
    print(f'pinned layout   -> layout_{lay_key}.npz   {coords.shape}')
    print(f'pinned clusters -> clusters_{cl_key}.npz  '
          f'{ {k: z[k].shape for k in z.files if k.startswith("layer")} }')
else:
    print('NO PINNED ARTIFACTS FOUND — the build will compute a fresh layout and the\n'
          'labels in this notebook will describe the WRONG clusters. See the note above.')

pinned layout   -> layout_3d44b484c3f99f3c.npz   (59579, 2)
pinned clusters -> clusters_1a4f330cc35617e5.npz  {'layer_0': (59579,), 'layer_1': (59579,), 'layer_2': (59579,)}


In [8]:
# ---- phase 1: build -----------------------------------------------------
# Embeddings, 2D layout and the 3-level cluster hierarchy, all cached in WORKDIR/cache
# keyed on the data hash + params. First run ~3 min (the layout dominates); re-runs
# with identical inputs are seconds.
cmd = [sys.executable, PIPELINE, 'build', '--data', CORPUS, '--text-col', TEXT_COL,
       '--workdir', WORKDIR, '--layers', str(LAYERS), '--seed', str(SEED),
       '--cluster-ks', CLUSTER_KS]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout or '', end=''); print(r.stderr[-2000:] if r.returncode else '', end='')
assert r.returncode == 0, 'build failed'

[datamap 18:53:34] full engine deps unavailable (sklearn/umap/hdbscan) — using LITE engine. Layout will be PCA-based; install deps for UMAP-quality maps.
[datamap 18:53:34] embeddings: cache hit (/tmp/dm_suricata/cache/embeddings_59b55e9358096376.npz)
[datamap 18:53:34] layout: cache hit (/tmp/dm_suricata/cache/layout_3d44b484c3f99f3c.npz)
[datamap 18:53:34] clusters: cache hit (/tmp/dm_suricata/cache/clusters_1a4f330cc35617e5.npz)
[datamap 18:53:34] summarizing clusters for labeling...
[datamap 18:53:39] build done in 5.0s. Now label clusters: read /tmp/dm_suricata/cluster_samples.json, write labels.json, then run `render`.


In [9]:
# ---- inspect the clusters before labeling -------------------------------
samples = json.load(open(f'{WORKDIR}/cluster_samples.json'))
print('engine:', samples.get('engine'), '| points:', f"{samples['n_points']:,}")
for layer in samples['layers']:
    print(f"\nlayer {layer['layer']}: {layer['n_clusters']} clusters, "
          f"noise {layer['noise_fraction']:.0%}")
    for c in layer['clusters'][:3]:
        print(f"   c{c['cluster_id']:<3} n={c['n_points']:<6} {c['sample_texts'][0][:96]}")

engine: lite | points: 59,579

layer 0: 12 clusters, noise 0%
   c0   n=5448   ET POLICY Powershell Command With No Profile Argument Over SMB - Likely Lateral Movement | A Net
   c1   n=8095   SURICATA TLS invalid record/traffic | Generic Protocol Command Decode | tls | 35302
   c2   n=3914   ET WEB_SERVER Possible CVE-2014-6271 Attempt in HTTP Cookie | Attempted Administrator Privilege 

layer 1: 48 clusters, noise 0%
   c0   n=1332   ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management | Not Suspiciou
   c1   n=2497   SURICATA TLS invalid record/traffic | Generic Protocol Command Decode | tls | 34895
   c2   n=793    ET WEB_SERVER Possible CVE-2014-6271 Attempt in Headers | Attempted Administrator Privilege Gain

layer 2: 144 clusters, noise 0%
   c0   n=276    ET POLICY GNU/Linux APT User-Agent Outbound likely related to package management | Not Suspiciou
   c1   n=1200   SURICATA TLS invalid record/traffic | Generic Protocol Command Decode | tls | 36021


## 3. Cluster labels

Written by hand from per-cluster field composition (dominant signatures, destination
ports, target hosts, URI vocabulary, source mix) — not from the c-TF-IDF keywords, which
in this corpus mostly surface tokenised URL fragments.

Labels suffixed `(A)`, `(B)`, … are **honest siblings**: several Suricata
protocol-anomaly families (TLS invalid records, TLS record version, SMTP no-welcome,
Heartbleed heartbeats, MySQL scanning) get split by the lite engine on the *digits of
the ephemeral destination port*, which carries no semantic signal. Their port
distributions overlap almost completely, so there is no real distinction to name and
inventing one would be false precision.

**These ids are only valid for `LAYERS=3`, `SEED=42`, `CLUSTER_KS=12,48,144`, lite
engine, and this exact corpus.** Change any of them and the labels will confidently
describe the wrong clusters.

In [10]:
LABELS = {
    'layer_0': {
        '0': 'Package Mirrors & Web Recon',
        '1': 'TLS Invalid Record Traffic',
        '2': 'Shellshock — Admin CGI',
        '3': 'Mixed TLS & SMTP Errors',
        '4': 'Dropbox Offsite Backup',
        '5': 'TLS Invalid Record Version',
        '6': 'Shellshock — Login & Bug Pages',
        '7': 'Shellshock — Form & Counter CGI',
        '8': 'Insecure Transfer Policy',
        '9': 'Heartbleed & HTTP Desync',
        '10': 'Service Scanning & Web Attacks',
        '11': 'SMTP Handshake Failures',
    },
    'layer_1': {
        '0': 'Debian Mirror Fetches',
        '1': 'TLS Invalid Records (A)',
        '2': 'Shellshock — wa.exe & Upload CGI',
        '3': 'TLS Errors on Port 443',
        '4': 'Dropbox Offsite Backup',
        '5': 'TLS Record Version (A)',
        '6': 'Shellshock — Board & Admin CGI',
        '7': 'Shellshock — /test & Buglist',
        '8': 'Executable Downloads',
        '9': 'Heartbleed Heartbeat Probes',
        '10': 'HTTP Host & DNS Anomalies',
        '11': 'SMTP No Welcome (A)',
        '12': 'MySQL Connection Scanning (A)',
        '13': 'Shellshock — Root & test-cgi',
        '14': 'HTTP Response Desync (A)',
        '15': 'ColdFusion & WordPress Probes',
        '16': 'Ubuntu Archive Fetches (A)',
        '17': 'PHP Info Leak & Odd User-Agents',
        '18': 'Shellshock — Admin & Store CGI',
        '19': 'SSLv3 & Cleartext Credentials',
        '20': 'Shellshock — Bugzilla & Cisco UCS',
        '21': 'Shellshock — NCBook & printenv',
        '22': 'TLS Invalid Handshake',
        '23': 'Shellshock — URI Payload',
        '24': 'Shellshock — LISTSERV wa.cgi',
        '25': 'Ubuntu Archive Fetches (B)',
        '26': 'TLS Invalid Records (B)',
        '27': 'Dropbox — Rare Ports',
        '28': 'Nmap — Metadata Probes',
        '29': 'Shellshock — cgi-mod & MT',
        '30': 'Shellshock — index & test.cgi',
        '31': 'TLS Record Version (B)',
        '32': 'SMTP & TLS Rejections',
        '33': 'SMTP No Welcome (B)',
        '34': 'TLS Invalid Records (C)',
        '35': 'Outdated Flash & Skype',
        '36': 'XSS & SQLi from 90.236.3.35',
        '37': 'TLS Errors — CDN Endpoints',
        '38': 'Nmap — Root & HNAP Probes',
        '39': 'Shellshock — mid.cgi & MailIt',
        '40': 'TLS Errors — Mozilla Endpoints',
        '41': 'Shellshock — index.sh & WHOIS',
        '42': 'Shellshock — Guestbook & ViewCVS',
        '43': 'Shellshock — Host 172.16.28.74',
        '44': 'Shellshock — Help & Bug Report',
        '45': 'Shellshock — Admin & Search CGI',
        '46': 'SMTP No Welcome (C)',
        '47': 'Shellshock — Counter Scripts',
    },
    'layer_2': {
        '0': 'Ubuntu Pool Packages',
        '1': 'TLS Invalid Records (A)',
        '2': 'Shellshock — Guestbook & Upload',
        '3': 'TLS Record Version on 443',
        '4': 'Dropbox — Rare Ports (A)',
        '5': 'TLS Record Version (A)',
        '6': 'Shellshock — cPanel & Login',
        '7': 'Shellshock — /test Path',
        '8': 'Vendor Executable Downloads',
        '9': 'Heartbleed Heartbeats (A)',
        '10': 'Invalid Host in Request URI',
        '11': 'SMTP No Welcome (A)',
        '12': 'MySQL Scanning (A)',
        '13': 'Shellshock — test-cgi Variants',
        '14': 'HTTP Response Desync (A)',
        '15': 'ColdFusion & WP Plugin Probes',
        '16': 'Ubuntu Archive Fetches (A)',
        '17': 'Dshield Drops & Poweliks CnC',
        '18': 'Shellshock — index.sh & index.pl',
        '19': 'Cleartext Password Submissions',
        '20': 'Shellshock — printenv & WP Login',
        '21': 'Shellshock — External Scanners',
        '22': 'TLS Invalid Handshake (A)',
        '23': 'Shellshock — hi & wa Paths',
        '24': 'Shellshock — administrator.cgi',
        '25': 'Ubuntu Archive Fetches (B)',
        '26': 'TLS Invalid Records (B)',
        '27': 'Dropbox — Rare Ports (B)',
        '28': 'XSS, SQLi & Struts RCE',
        '29': 'Shellshock — MT Static Scripts',
        '30': 'Shellshock — index.cgi & test.cgi',
        '31': 'TLS Record Version (B)',
        '32': 'SMTP Invalid Replies',
        '33': 'SMTP No Welcome (B)',
        '34': 'TLS Invalid Records (C)',
        '35': 'Flash, Skype & BitTorrent',
        '36': 'XSS Script Tag Probes',
        '37': 'TLS Errors — aspnetcdn',
        '38': 'Nmap — HNAP & ColdFusion',
        '39': 'Shellshock — Buglist & Query',
        '40': 'TLS Errors — Mozilla Endpoints',
        '41': 'Shellshock — WHOIS & XAMPP',
        '42': 'Shellshock — FAQ Manager',
        '43': 'Shellshock — spytellite8.com',
        '44': 'Shellshock — clwarn & FormHandler',
        '45': 'Shellshock — Admin Scripts',
        '46': 'SMTP No Welcome (C)',
        '47': 'Shellshock — admin.cgi',
        '48': 'Nmap — .git Probes',
        '49': 'PHP Easter Egg Disclosure (A)',
        '50': 'Debian Mirrors & Heartbeat',
        '51': 'Dropbox Backup Sessions (A)',
        '52': 'Shellshock — Bugzilla & UCS',
        '53': 'Nmap — Root Path Probes',
        '54': 'Shellshock — Board CGI',
        '55': 'Shellshock — QuickStore & Forms',
        '56': 'TLS Invalid Records — Fragment',
        '57': 'Ubuntu Security Updates',
        '58': 'Shellshock — wa.cgi Headers',
        '59': 'Dropbox — Small Port Group',
        '60': 'Shellshock — Root Path',
        '61': 'TLS Invalid Records on 443',
        '62': 'Basic Auth & YUM Traffic',
        '63': 'Shellshock — Search & index CGI',
        '64': 'Nmap — SDK & Hadoop Endpoints',
        '65': 'Debian Security Updates',
        '66': 'Ubuntu LibreOffice Packages (A)',
        '67': 'Shellshock — spytellite Hosts',
        '68': 'Dropbox — Single Session',
        '69': 'TLS Invalid Records (D)',
        '70': 'Shellshock — Help & URL Count',
        '71': 'TLS Record Version (C)',
        '72': 'DNS Response Anomalies',
        '73': 'Shellshock — wa.exe & MT Load',
        '74': 'Dropbox — Port Group (A)',
        '75': 'Shellshock — LISTSERV URI Payload',
        '76': 'Shellshock — test-cgi URI',
        '77': 'Nmap — Hadoop & robots.txt',
        '78': 'Heartbleed Heartbeats (B)',
        '79': 'Shellshock — NCBook & EntropySearch',
        '80': 'Nmap — Favicon & Directory Probes',
        '81': 'TLS Errors — Ad & Telemetry',
        '82': 'Shellshock — Counter Scripts',
        '83': 'PHP CGI Injection Probes',
        '84': 'Dropbox — Port Trio',
        '85': 'Dropbox — Port Group (B)',
        '86': 'Shellshock — admin.pl & test.sh',
        '87': 'TLS Invalid Records (E)',
        '88': 'Heartbleed Probes — OCF Mirror',
        '89': 'Heartbleed Heartbeats (C)',
        '90': 'Ubuntu Archive Fetches (C)',
        '91': 'Shellshock — MT Admin Console',
        '92': 'TLS Invalid Records — Fragment (B)',
        '93': 'TLS Errors — CDN & Twitter',
        '94': 'TLS & SMTP Errors — ise.wrccdc.org',
        '95': 'MySQL Scanning (B)',
        '96': 'SMTP TLS Rejections',
        '97': 'Shellshock — Host 172.16.28.74',
        '98': 'SQLi & Directory Traversal',
        '99': 'TLS Record Version (D)',
        '100': 'HTTP Response Desync (B)',
        '101': 'Shellshock — session_login.cgi',
        '102': 'Shellshock — test.cgi & Search',
        '103': 'TLS Record Version (E)',
        '104': 'Shellshock — index.cgi & test.sh',
        '105': 'Shellshock — index Scripts',
        '106': 'TLS Invalid Records (F)',
        '107': 'Drupalgeddon & IIS DoS',
        '108': 'Mixed: phpinfo & Package Fetches',
        '109': 'HTTP Response Desync (C)',
        '110': 'TLS Invalid Records (G)',
        '111': 'MySQL Scanning (C)',
        '112': 'SMTP No Welcome (D)',
        '113': 'Windows Update & Credentials',
        '114': 'Package Fetches & Installer UAs',
        '115': 'Ambiguous HTTP Host Headers',
        '116': 'Shellshock — clwarn & Upload',
        '117': 'Shellshock — mid.cgi & MailIt',
        '118': 'TLS Invalid Records (H)',
        '119': 'Minecraft Launcher & SMB Jobs',
        '120': 'Debian & Kali Repo Fetches',
        '121': 'MySQL Scanning (D)',
        '122': 'Dropbox — Port Quad',
        '123': 'TLS Handshake Errors on 443',
        '124': 'Dropbox Backup Sessions (B)',
        '125': 'Heartbleed Heartbeats (D)',
        '126': 'SMTP No Welcome (E)',
        '127': 'XSS Probes & Shell Responses',
        '128': 'PHP Easter Egg Disclosure (B)',
        '129': 'SSLv3 to Windows Update',
        '130': 'Mixed: S3 Executables & SMB',
        '131': 'Malformed HTTP Methods & SLP',
        '132': 'Basic Auth & VNC Scanning',
        '133': 'SMTP No Welcome (F)',
        '134': 'Rsyslog & Firefox Downloads',
        '135': 'Dropbox — Port Group (C)',
        '136': 'TLS Record Version (F)',
        '137': 'TLS Errors — Microsoft & Google',
        '138': 'Shellshock — cgi-mod & Search',
        '139': 'Dropbox — Single Session (B)',
        '140': 'Invalid Host Headers — Zabbix',
        '141': 'Heartbleed Heartbeats (E)',
        '142': 'SMTP No Welcome (G)',
        '143': 'Ubuntu LibreOffice Packages (B)',
    },
}

LABELS_PATH = '/tmp/labels.json'
json.dump(LABELS, open(LABELS_PATH, 'w'), indent=2)
print({k: len(v) for k, v in LABELS.items()}, '=', sum(len(v) for v in LABELS.values()), 'labels')

# every cluster in every layer must be labelled -- render refuses otherwise
import numpy as np, glob
z = np.load(glob.glob(f'{WORKDIR}/cache/clusters_*.npz')[0])
for i in range(LAYERS):
    ids = {str(c) for c in set(z[f'layer_{i}'].tolist()) - {-1}}
    missing = ids - set(LABELS[f'layer_{i}'])
    print(f'layer_{i}: {len(ids)} clusters, {len(missing)} unlabelled')
    assert not missing, missing

{'layer_0': 12, 'layer_1': 48, 'layer_2': 144} = 204 labels
layer_0: 12 clusters, 0 unlabelled
layer_1: 48 clusters, 0 unlabelled
layer_2: 144 clusters, 0 unlabelled


In [11]:
# ---- phase 2: render ----------------------------------------------------
ROWS = f'{len(df):,}'
SUBTITLE = f'WRCCDC 2017/2018 · FIRST 2015 · Honeypot 2018 · n={ROWS}'

cmd = [sys.executable, PIPELINE, 'render', '--workdir', WORKDIR, '--labels', LABELS_PATH,
       '--out', OUT_HTML, '--title', TITLE, '--subtitle', SUBTITLE,
       '--mode', 'custom', '--color-col', COLOR_COL, '--palette', PALETTE,
       '--hover-cols', HOVER_COLS, '--record-cols', RECORD_COLS,
       '--card-link-col', 'signature_id',
       '--card-link-url', SID_LOOKUP_URL.replace('{sid}', '{v}'),
       '--card-link-label', 'ET rule']
if DARKMODE:
    cmd += ['--darkmode']
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout or '', end=''); print(r.stderr[-2000:] if r.returncode else '', end='')
assert r.returncode == 0, 'render failed'

[datamap 18:53:52] palette overridden from /tmp/palette.json
[datamap 18:53:53] columnar payload: 59,579 points x 14 fields (9 dictionary-encoded)
[datamap 18:53:53] card link: signature_id -> https://threatintel.proofpoint.com/sid/{v} (100.0% of points)
[datamap 18:53:56] preview PNG written (matplotlib fallback): /tmp/suricata_alert_datamap_2026-08-12_preview.png
[datamap 18:53:56] wrote /tmp/suricata_alert_datamap_2026-08-12.html (8.9 MB) and /tmp/suricata_alert_datamap_2026-08-12_preview.png


In [12]:
# ---- validate -----------------------------------------------------------
r = subprocess.run([sys.executable, PIPELINE, 'validate', '--workdir', WORKDIR],
                   capture_output=True, text=True)
print(r.stdout, end='')

import re as _re
html = open(OUT_HTML, encoding='utf-8').read()
payload = json.loads(_re.search(r'const DATA = (\{.*?\});\n', html, _re.S).group(1))
def _col(c, i):
    return c['d'][c['i'][i]] if 'd' in c else c['v'][i]

npts = len(payload['points'])
print('\nHTML size          :', f'{os.path.getsize(OUT_HTML)/1e6:.1f} MB')
print('payload keys       :', sorted(payload))
print('points             :', f'{npts:,}')
print('clusters per layer :', [len(l) for l in payload['label_layers']])
print('legend             :', [(e['label'], e['color'], e['n']) for e in payload['legend']])
print('\ncolumn encoding (d = dictionary, v = raw):')
for c, col in payload['cols'].items():
    kind = f"d[{len(col['d']):,}]" if 'd' in col else 'v'
    print(f'  {c:11s} {kind:>10s}  {len(json.dumps(col))/1e6:6.2f} MB')
print(f"  {'fine':11s} {'d[%d]' % len(payload['fine']['d']):>10s}  "
      f"{len(json.dumps(payload['fine']))/1e6:6.2f} MB")

# every column must cover every point
for c, col in payload['cols'].items():
    assert len(col['i'] if 'i' in col else col['v']) == npts, f'{c} length mismatch'
assert len(payload['fine']['i']) == npts
assert payload['legend'] and payload['colors']
assert [len(l) for l in payload['label_layers']] == [12, 48, 144]
assert 'hover' not in payload and 'records' not in payload, 'stale bulk keys still present'
unlabelled = sum(1 for i in range(npts) if _col(payload['fine'], i) == 'Unlabelled')
print('\nunlabelled points  :', unlabelled)

# Regression guard: the corpus contains XSS-probe URIs with a literal '</script>'.
# Injected unescaped, the first one ends the script element and the page never runs.
n_close = len(_re.findall(r'</script\s*>', html, _re.I))
n_open  = len(_re.findall(r'<script[ >]', html, _re.I))
print(f'\nscript tags in document: {n_open} open / {n_close} close (must be 1 / 1)')
assert n_close == 1 and n_open == 1, 'payload broke out of the <script> element'
for bad in ('<', '>'):
    assert bad not in html[html.index('const DATA = '):html.index('</script>')].split('\n')[0], \
        f'unescaped {bad!r} survived in the payload line'
print('payload is inert to the HTML tokenizer')
print('\nall payload assertions passed')

  PASS  manifest.json exists
  PASS  build stage recorded in manifest
  PASS  layout coordinates all finite
  PASS  layout is not degenerate (points not collapsed)
  PASS  layer 0 has >=1 cluster (12 found)
  PASS  layer 1 has >=1 cluster (48 found)
  PASS  layer 2 has >=1 cluster (144 found)
  PASS  HTML output is non-trivial (8,933,695 chars)
  PASS  HTML contains renderer code
  PASS  preview PNG exists and is non-trivial
VALIDATION PASSED

HTML size          : 8.9 MB
payload keys       : ['card_link', 'colors', 'cols', 'darkmode', 'fine', 'hover_cols', 'hover_max_chars', 'label_layers', 'legend', 'legend_title', 'points', 'subtitle', 'title', 'urls']
points             : 59,579
clusters per layer : [12, 48, 144]
legend             : [('first-org-2015', '#ffd166', 4429), ('honeypot-2018', '#ff2e63', 150), ('wrccdc-2017', '#4cc9f0', 27500), ('wrccdc-2018', '#b388ff', 27500)]

column encoding (d = dictionary, v = raw):
  signature       d[293]    0.29 MB
  signature_id     d[294]    0

In [13]:
# ---- preview ------------------------------------------------------------
from IPython.display import Image, HTML, display
png = os.path.splitext(OUT_HTML)[0] + '_preview.png'
display(Image(filename=png))
display(HTML(f'<a href="{OUT_HTML}" target="_blank">open interactive datamap →</a>'))

<Image /tmp/suricata_alert_datamap_2026-08-12_preview.png (345 KB)>
<HTML <a href="/tmp/suricata_alert_datamap_2026-08-12.html" target="_blank">open inter>


In [14]:
# ---- stamp provenance into the manifest ---------------------------------
mf_path = f'{WORKDIR}/manifest.json'
mf = json.load(open(mf_path))
mf['viewer_template_source'] = TEMPLATE          # patched click-to-open-card viewer
mf['pipeline_patched'] = True                    # stock skill script + the 8 subs above
mf['corpus'] = {
    'rows': int(len(df)),
    'raw_alerts_represented': int(df['count'].astype(int).sum()),
    'unique_texts': int(df['text'].nunique()),
    'unique_signatures': int(df['signature'].nunique()),
    'uri_nonce_normalized': True,
    'per_source': {s['source']: {k: s[k] for k in
                   ('alerts', 'unique_composites', 'unique_signatures', 'rows_kept')}
                   for s in stats},
}
mf['source_dataset'] = 'https://github.com/FrankHassanabad/suricata-sample-data/releases/tag/v4.0.0'
json.dump(mf, open(mf_path, 'w'), indent=2)
print('manifest stamped:')
print('  template   :', mf['viewer_template_source'])
print('  cluster_ks :', mf['build_params'].get('cluster_ks'))
print('  record_cols:', mf['render_params'].get('record_cols'))

manifest stamped:
  template   : /sessions/happy-clever-brown/mnt/Suricata - Datamap/suricata_datamap_template.html
  cluster_ks : 12,48,144
  record_cols: signature,signature_id,category,severity,src_ip,src_port,dest_ip,dest_port,app_proto,hostname,uri,count,timestamp,source


In [15]:
# ---- copy deliverables into the shared folder ---------------------------
for src_path, name in [
    (OUT_HTML,  f'suricata_alert_datamap_{DATE}.html'),
    (png,       f'suricata_alert_datamap_{DATE}_preview.png'),
    (CORPUS,    'suricata_alerts_corpus.csv'),
    (LABELS_PATH, 'labels.json'),
    (f'{WORKDIR}/manifest.json', 'manifest.json'),
]:
    shutil.copy(src_path, os.path.join(BASE, name))
    print(f'{name:46s} {os.path.getsize(src_path)/1e6:7.2f} MB')

suricata_alert_datamap_2026-08-12.html            8.93 MB
suricata_alert_datamap_2026-08-12_preview.png     0.34 MB
suricata_alerts_corpus.csv                       20.02 MB
labels.json                                       0.01 MB
manifest.json                                     0.00 MB


## 4. Package as a hostable PWA

`build_pwa.py` splits the rendered single-file HTML into an app shell plus a fetched data
file and adds the PWA scaffolding (manifest, service worker, icons drawn from the real
layout coordinates). The single-file HTML is left untouched — it stays the portable
`file://` artifact; the PWA is additive and must be served over HTTP(S).

Splitting matters at this size: precaching one 8.5 MB file means every viewer tweak
re-downloads the whole payload. Split, the shell (27 KB) and the data (7.8 MB, 1.29 MB
gzipped) are cached separately and the cache name carries a hash of both.

In [16]:
# ---- build the PWA bundle -----------------------------------------------
r = subprocess.run([sys.executable, f'{BASE}/build_pwa.py'],
                   capture_output=True, text=True, env={**os.environ,
                                                        'SURICATA_DATAMAP_DIR': BASE})
print(r.stdout or '', end=''); print(r.stderr[-2000:] if r.returncode else '', end='')
assert r.returncode == 0, 'PWA build failed'

building PWA bundle
  data/datamap.json           7.80 MB (59,579 points)
  index.html                  27.4 KB (shell only)
  icon: 3,533 points, coloured by source
  icons/icon-192.png          25.7 KB
  icons/icon-512.png          96.7 KB
  icons/icon-180.png          23.2 KB
  icons/icon-512-maskable.png    58.9 KB
  manifest.webmanifest
  preview.png
  sw.js                    cache version 999e472f29b6
  .nojekyll

  bundle total: 8.38 MB in /sessions/happy-clever-brown/mnt/Suricata - Datamap/pwa


In [17]:
# ---- verify the bundle --------------------------------------------------
import re as _re2
PWA = f'{BASE}/pwa'
sw = open(f'{PWA}/sw.js', encoding='utf-8').read()
assets = json.loads(_re2.search(r'const ASSETS = (\[.*?\]);', sw, _re2.S).group(1))
missing = [a for a in assets
           if not os.path.exists(os.path.join(PWA, 'index.html' if a == './' else a))]
print('precached assets   :', len(assets), '| missing:', missing or 'none')
assert not missing, 'service worker install would 404 and never activate'

mf = json.load(open(f'{PWA}/manifest.webmanifest', encoding='utf-8'))
print('manifest           :', mf['name'], '|', mf['display'], '| icons:', len(mf['icons']))
assert mf['start_url'] == './' and mf['scope'] == './', 'relative scope required for /repo/ paths'
assert any(i.get('purpose') == 'maskable' for i in mf['icons']), 'need a maskable icon'

shell = open(f'{PWA}/index.html', encoding='utf-8').read()
assert '__DATAMAP_DATA__' not in shell and 'const DATA = {' not in shell, 'data still inlined'
assert 'serviceWorker' in shell and 'rel="manifest"' in shell
assert not _re2.search(r'(href|src)="/(?!/)', shell), 'absolute path breaks project subpaths'
print('shell              :', f'{os.path.getsize(f"{PWA}/index.html")/1e3:.1f} KB, no inlined data')

import gzip as _gz
raw = open(f'{PWA}/data/datamap.json', 'rb').read()
comp = len(_gz.compress(raw, 6))
d = json.loads(raw)
print(f'payload            : {len(raw)/1e6:.2f} MB raw -> {comp/1e6:.2f} MB gzipped '
      f'({comp/len(raw)*100:.0f}%), {len(d["points"]):,} points')
assert len(d['points']) == len(df), 'payload lost points'
total = sum(os.path.getsize(os.path.join(r_, f)) for r_, _, fs in os.walk(PWA) for f in fs)
print(f'bundle total       : {total/1e6:.2f} MB')

precached assets   : 8 | missing: none
manifest           : Suricata Alert Datamap | standalone | icons: 3
shell              : 27.4 KB, no inlined data
payload            : 7.80 MB raw -> 1.29 MB gzipped (17%), 59,579 points
bundle total       : 8.38 MB


## Limitations and traps

**1. The composite text field is load-bearing.** Reduce `TEXT_COL` to `signature` alone
and the map collapses: measured on the full captures there are only **43** distinct
signatures in honeypot-2018, **135** in first-org-2015, **161** in wrccdc-2017 and
**206** in wrccdc-2018 — 293 across the whole sampled corpus. The composite
`signature | category | app_proto | hostname | uri | dest_port` is what turns ~300
distinct strings into 58,787 distinct positions.

**2. …but the raw URI manufactures fake variety, which is the opposite trap.** The
NETGEAR WNR2000v5 exploit URI carries a random `timestamp=<8 digits>` nonce. Without
`normalize_uri`, dedup cannot collapse it and **22,173 rows — 37% of the corpus — are
one alert** (21 real variants) shattered across ~10 meaningless clusters, crowding out
everything else. Normalising long digit runs and hex blobs in the dedup key cut
wrccdc-2018 from 463,048 "unique" composites to 89,068 real ones. The raw `uri` column
is untouched and is still what the tooltip and info card display. *If you widen the
nonce pattern you will over-collapse; if you remove it the map reverts to a NETGEAR
monoculture.*

**3. Ephemeral destination ports are the residual noise.** `dest_port` is kept in the
composite because for SNMP/TFTP/telnet/MySQL scan traffic it is the only discriminator.
For Suricata protocol-anomaly alerts, though, it is the client's random high port, so
those families fragment into siblings that differ by nothing but digits — hence the
`(A)`/`(B)` labels. Normalising ports ≥10000 as well would collapse the corpus to 13,514
rows; that was judged too destructive, since it also flattens genuine scan targets.

**4. Labels are pinned to the shipped ARTIFACTS, not just the parameters.** This is the
sharpest trap in the whole notebook. Same params, same corpus bytes, different process →
different map: `lite_embed`'s randomized SVD runs through multithreaded BLAS, whose
reduction order is not fixed, so a rerun moved the embeddings by up to 1.52 and a measured
rebuild agreed with the previous per-point labels only **5%** of the time. Rounded
coordinates are not enough to recover it either — re-clustering from the 3-decimal coords
in a rendered HTML gave 8.5% agreement, because k-means diverges over 60 iterations from a
5e-4 perturbation. Hence `layout_coords.npy` + `cluster_assignments.npz` are deliverables
and the pinning cell above is mandatory. Changing `LAYERS`, `SEED`, `CLUSTER_KS`, or
installing umap/hdbscan so the *full* engine activates, all invalidate the labels too —
rebuild and relabel together, or the map lies.

**5. The payload must be escaped before it is injected into `<script>`.** The stock
`render_custom()` writes `json.dumps(payload)` straight into the script element. `json.dumps`
does not escape `<`, and this corpus contains 573 XSS-probe URIs holding a literal
`</script>`. The HTML parser closes the script at the first one — about 1% into the file —
so the viewer never initialises and the browser tries to lay out the remaining ~33 MB of
JSON as text, which looks exactly like "the file won't open". Patch (8) escapes `<`, `>`,
`&`, U+2028 and U+2029 as `\uXXXX`; the JSON is semantically unchanged and JS parses it
back identically. The validate cell asserts the document contains exactly one
`<script>`/`</script>` pair — keep that assertion, it is the only cheap way to catch a
regression here short of opening a browser.

**6. The template patch is not optional.** `render_custom()` hardcodes
`<skill>/assets/custom_template.html`. The patched viewer must be copied into place
before rendering; skip it and you get the stock viewer, no info cards, no legend, and no
warning.

**7. Any field the info card shows must stay in `RECORD_COLS`.** The card reads
`payload.cols` only. Dropping a column from `RECORD_COLS` makes that row silently vanish
from the card. `src_port` is in `RECORD_COLS` but deliberately not in `HOVER_COLS` — the
card needs it to render `src→dst` with ports, the tooltip does not. `HOVER_COLS` must be a
subset of `RECORD_COLS`, because the tooltip is now assembled in the browser from those
same columns; a hover column that is not shipped renders as `undefined`.

**8. Source sizes differ by four orders of magnitude, so sampling is deliberate, not
proportional.** honeypot-2018 (254 alerts) and first-org-2015 (7,750) are kept whole;
the two WRCCDC captures are sampled to 27,500 deduplicated composites each. Proportional
sampling would have erased the honeypot entirely. The consequence: **cluster sizes on
this map are not incidence rates.** The `count` column carries how many raw alerts
collapsed into each point (496,578 in total) — use that, not point density, for volume.

**9. The PWA cannot run from `file://`, and the single-file HTML can.** They are two
different artifacts on purpose. `pwa/index.html` fetches `data/datamap.json`, which the
`file://` origin blocks; the shell detects that and says so rather than failing blank.
Service workers additionally require a secure origin, so the PWA needs HTTPS (or
`localhost`). Hosts that serve `.webmanifest` as `application/octet-stream` — nginx and S3
by default — will make Chrome refuse to install the app; see the README. Keep
`Cache-Control: no-cache` on `sw.js` only, or browsers will pin an old service worker.

**10. The lite engine is a fallback, not a choice.** No outbound network here, so
UMAP/HDBSCAN/datamapplot could not be installed. The layout is PCA + neighbour
refinement and the hierarchy is plain k-means, which produces straighter, more banded
structure than UMAP and has no noise class (`noise_fraction` is 0 by construction —
every point is forced into a cluster). Re-running with the full engine installed will
change the layout and every cluster id.